In [2]:
# !pip install unicorn_binance_websocket_api

In [54]:
import json
from websocket import WebSocketApp
import ssl
import pandas as pd

In [55]:
WS_URL = "wss://stream.bybit.com/v5/public/spot"

In [56]:
df = pd.DataFrame(columns=["Timestamp", "open", "high", "low", "close", "volume"])

In [57]:
def on_message(ws, message):
    data = json.loads(message)

    if "topic" in data and data["topic"].startswith("kline"):
        k = data["data"][0]
        ts = (
        pd.to_datetime(k['timestamp'], unit='ms', utc=True)
        .tz_convert('Asia/Kolkata')
        .strftime('%Y-%m-%d %H:%M:%S')
      )
        candle = {
            "Timestamp": ts,   # <-- FIXED!!!
            "open": float(k["open"]),
            "high": float(k["high"]),
            "low": float(k["low"]),
            "close": float(k["close"]),
            "volume": float(k["volume"]),
        }

        global df
        df = pd.concat([df, pd.DataFrame([candle])], ignore_index=True)

        print("\n---- NEW CANDLE ----")
        print(df.tail(1))
        print("--------------------")

In [58]:
def on_open(ws):
    print("Connected.")
    sub = {"op": "subscribe", "args": ["kline.60.BTCUSDT"]}
    ws.send(json.dumps(sub))
    print("Subscription sent.")

def on_close(ws, a, b):
    print("Connection closed.")

def on_error(ws, error):
    print("Error:", error)

In [59]:
ws = WebSocketApp(
    WS_URL,
    on_open=on_open,
    on_message=on_message,
    on_close=on_close,
    on_error=on_error,
)

ws.run_forever(sslopt={"cert_reqs": ssl.CERT_NONE})

Connected.
Subscription sent.

---- NEW CANDLE ----
             Timestamp     open     high      low    close      volume
0  2025-12-11 16:01:54  90228.0  90346.9  90163.4  90204.1  123.209563
--------------------


/tmp/ipython-input-635769081.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([candle])], ignore_index=True)



---- NEW CANDLE ----
             Timestamp     open     high      low    close      volume
1  2025-12-11 16:01:55  90228.0  90346.9  90163.4  90204.1  123.209578
--------------------

---- NEW CANDLE ----
             Timestamp     open     high      low    close      volume
2  2025-12-11 16:01:58  90228.0  90346.9  90163.4  90219.6  123.511206
--------------------

---- NEW CANDLE ----
             Timestamp     open     high      low    close      volume
3  2025-12-11 16:01:59  90228.0  90346.9  90163.4  90222.0  123.511475
--------------------

---- NEW CANDLE ----
             Timestamp     open     high      low    close      volume
4  2025-12-11 16:02:01  90228.0  90346.9  90163.4  90218.4  124.130923
--------------------

---- NEW CANDLE ----
             Timestamp     open     high      low    close      volume
5  2025-12-11 16:02:02  90228.0  90346.9  90163.4  90216.7  124.131023
--------------------

---- NEW CANDLE ----
             Timestamp     open     high      low    

True

In [62]:
df.head()

,Timestamp,open,high,low,close,volume
0,2025-12-11 16:01:54,90228.0,90346.9,90163.4,90204.1,123.209563
1,2025-12-11 16:01:55,90228.0,90346.9,90163.4,90204.1,123.209578
2,2025-12-11 16:01:58,90228.0,90346.9,90163.4,90219.6,123.511206
3,2025-12-11 16:01:59,90228.0,90346.9,90163.4,90222.0,123.511475
4,2025-12-11 16:02:01,90228.0,90346.9,90163.4,90218.4,124.130923


In [61]:
df.to_csv('bybit_btcusdt.csv', index=False)